In [1]:
# Imports
import numpy as np
import pandas as pd

In [2]:
# Load the dataset
df = pd.read_csv("scraped_data.csv")

C:\Users\ollie\AppData\Local\Temp\ipykernel_27148\969364503.py:2: DtypeWarning: Columns (0: 0-60-0mph / Gravel, 1: 0-60mph / Gravel, 2: Desert Rally Hill (C) / Aspht Dirt, 3: Lumber Mill Twisty Circuit / Aspht Dirt Wet, 4: Mountain Twisty Road / Dirt, 5: Test Bowl / Dry, 6: Test Bowl / Dirt, 7: Test Bowl (R) / Dry, 8: Test Bowl (R) / Snow) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('scraped_data.csv')


In [3]:
int_cols = [
    "rq",
    "year",
    "engine_up",
    "weight_up",
    "chassis_up",
    "weight",
    "seats",
]
for col in int_cols:
    df[col] = df[col].astype("int")

In [4]:
def convert_to_seconds(time_str):
    try:
        time_in_seconds = 0
        time_in_seconds += float(time_str.split(":")[0]) * 60  # minutes to seconds
        time_in_seconds += float(time_str.split(":")[1])  # seconds
        time_in_seconds += float(time_str.split(":")[2]) / 100  # deci-seconds
        return round(float(time_in_seconds), 2)
    except:
        return np.inf

In [5]:
def process_test_bowl(speed_str):
    try:
        if int(speed_str) > 1:
            return int(speed_str)
        else:
            return 0
    except:
        return 0

In [6]:
# Convert all times to seconds
# Iterate through columns
for col in df.columns:
    # If we get to rq column, break
    if col == "rq":
        break

    # If track is a test bowl, skip it
    if "Test Bowl" in col:
        df[col] = df[col].apply(process_test_bowl)
    # Else if the column is a time column, convert it to seconds
    else:
        try:
            df[col] = df[col].apply(convert_to_seconds)
        except:
            print(f"Error converting column {col} to seconds")

In [7]:
# Convert info columns to one hot encoding
cols_to_encode_simple = [
    "prize",
    "country",
    "tyres",
    "drive",
    "abs",
    "tcs",
    "clearance",
    "fuel",
    "seats",
    "engine_pos",
    "brakes",
]

# Make copy of df
df_one_hot = df.copy()

# Encode simple columns
for col in cols_to_encode_simple:
    df_one_hot[col] = df_one_hot[col].astype("category")
    df_one_hot = pd.get_dummies(df_one_hot, columns=[col], prefix=col, dtype="int")

# Encode complex columns
# Get all unique tags
all_tags = set()
for joined_tags in df["tags"].unique():
    tags = joined_tags.split("/")
    for tag in tags:
        all_tags.add(tag)

# Encode tags
for tag in all_tags:
    df_one_hot[f"tag_{tag.replace(' ', '_')}"] = df["tags"].apply(
        lambda x: 1 if tag in x.split("/") else 0
    )
df_one_hot.drop(columns=["tags"], inplace=True)

# Get all unique body types
all_bodies = set()
for joined_bodies in df["body"].unique():
    bodies = joined_bodies.split("/")
    for body in bodies:
        all_bodies.add(body)
# Encode body types
for body in all_bodies:
    df_one_hot[f"body_{body.replace(' ', '_')}"] = df["body"].apply(
        lambda x: 1 if body in x.split("/") else 0
    )
df_one_hot.drop(columns=["body"], inplace=True)

# Get rarity flags
rarity_ranges = {
    "S": (80, 120),
    "A": (65, 80),
    "B": (50, 65),
    "C": (40, 50),
    "D": (30, 40),
    "E": (20, 30),
    "F": (0, 20),
}
for r in ["S", "A", "B", "C", "D", "E", "F"]:
    df_one_hot[f"rarity_{r}"] = df["rq"].apply(
        lambda x: 1 if rarity_ranges[r][0] <= x < rarity_ranges[r][1] else 0
    )

In [8]:
# unidecode make, model, make_model
from unidecode import unidecode

df_one_hot["make"] = df["make"].apply(lambda x: unidecode(x))
df_one_hot["model"] = df["model"].apply(lambda x: unidecode(x).replace("&amp;", "&"))
df_one_hot["make_model"] = df["make_model"].apply(lambda x: unidecode(x).replace("&amp;", "&"))

In [9]:
# Save encoded dataframe to csv
df_one_hot.to_csv("scraped_data_encoded.csv", index=False)